#EVALUATORS

In [1]:
from langsmith.schemas import Example, Run

def correct_label(inputs: dict, reference_outputs: dict, outputs: dict) -> dict:
  score = outputs.get("output") == reference_outputs.get("label")
  return {"score": int(score), "key": "correct_label"}

In [5]:
env_path = find_dotenv()
load_dotenv(env_path, override=True)

True

#LOW SCORE CODE

In [18]:
import os
import re
import requests
from pydantic import BaseModel, Field
PPLX_API_KEY = os.environ.get("PPLX_API_KEY")
class Similarity_Score(BaseModel):
    similarity_score: int = Field(..., description="Semantic similarity score between 1 and 10, where 1 means unrelated and 10 means identical.")
def _call_perplexity(messages, model="sonar-pro", temperature=0.0, timeout=60):
    payload = {"model": model, "messages": messages, "temperature": temperature}
    resp = requests.post(
        BASE_URL,
        headers={"Authorization": f"Bearer {PPLX_API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=timeout,
    )
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data.get("choices"), list) and data["choices"]:
        choice = data["choices"][0]
        # chat-style nested message
        if isinstance(choice.get("message"), dict):
            return choice["message"].get("content", "")
        return choice.get("text", "") or ""
    if "text" in data:
        return data["text"]
    return str(data)
def _parse_score(text: str) -> int:
    m = re.search(r"\b([1-9]|10)\b", text)
    if m:
        return int(m.group(1))
    m2 = re.search(r"(\d{1,2})", text)
    if m2:
        n = int(m2.group(1))
        return max(1, min(10, n))
    return 5
def compare_semantic_similarity(inputs: dict, reference_outputs: dict, outputs: dict):
    question = inputs.get("question", "")
    reference = reference_outputs.get("output", "")
    run = outputs.get("output", "")
    system = (
        "You are an evaluator that measures how closely two answers match in meaning. "
    "You will be given a Reference Answer (the correct one) and a Candidate Answer (the new model output). "
    "Score their semantic similarity on an integer scale from 1 to 10, where 1 means completely different and 10 means identical in intent and meaning. "
    )
    user = f"Question: {question}\nReference Response: {reference}\nNew Response: {run}"

    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]

    text = _call_perplexity(messages, model="sonar-pro", temperature=0.0)
    score = _parse_score(text)

    validated = Similarity_Score(similarity_score=score)
    return {"score": validated.similarity_score, "key": "similarity"}
if __name__ == "__main__":
    result = compare_semantic_similarity(example_inputs, example_reference, example_run)
    print("Similarity result:", result)


Similarity result: {'score': 2, 'key': 'similarity'}


In [20]:
inputs = {
    "question": "What does this diagram show?"
}
reference_outputs = {
    "output": "The diagram illustrates a simple RAG pipeline with retrieval, context integration, and final answer generation."
}
outputs = {
    "output": "It shows a process where information is fetched, combined, and then used to produce the answer."
}
similarity_score = compare_semantic_similarity(inputs, reference_outputs, outputs)
print(f"Semantic similarity score: {similarity_score}")


Semantic similarity score: {'score': 9, 'key': 'similarity'}


#DEFINING EVALUATORS USING RUN AND EXAMPLE

In [22]:
def compare_semantic_similarity_v2(run: dict, example: dict):
    inputs = run["inputs"]
    reference_outputs = example["outputs"]
    outputs = run["outputs"]
    return compare_semantic_similarity(inputs, reference_outputs, outputs)


In [23]:
sample_run = {
    "name": "Sample Run",
    "inputs": {
        "question": "What does this diagram represent?"
    },
    "outputs": {
        "output": "It shows a simple flow of retrieving information and then generating an answer."
    },
    "is_root": True,
    "status": "success",
    "extra": {
        "metadata": {
            "source_image": "/mnt/data/cd0facce-3b15-462e-9c3b-421a92e08d74.png"
        }
    }
}

sample_example = {
    "inputs": {
        "question": "What does this diagram represent?"
    },
    "outputs": {
        "output": "The diagram represents a retrieval-augmented generation pipeline with three steps: retrieve, combine context, and answer."
    },
    "metadata": {
        "dataset_split": [
            "AI generated",
            "reference"
        ]
    }
}

similarity_score = compare_semantic_similarity_v2(sample_run, sample_example)
print(f"Semantic similarity score: {similarity_score}")


Semantic similarity score: {'score': 6, 'key': 'similarity'}
